In [1]:
"""
Main Engine (Threshold-free Stability/Transition Engine)
- Dataset: MHEALTH (.log)
- Multi-sensor/modalities with missing groups supported
- Detector: per-group steady score s_t in [0,1] (threshold-free)
- Sensor fusion: self-consistency quality -> softmax weights
- Model: transition probability sequence + latent + reconstruction
- Training: recon + soft transition supervision + pair consistency contrast + inertial invariance (optional)
- (Optional) Count K supervision is included as a hook, but NOT used at test-time.
"""

import os, glob, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from scipy.signal import savgol_filter

# =========================================================
# 0) CONFIG
# =========================================================
CONFIG = {
    # data
    "data_dir": "/content/drive/MyDrive/Colab Notebooks/HAR_data/MHEALTHDATASET",
    "target_activities": [6, 7, 11],   # arms / knees / running
    "fs": 50,

    # windowing
    "window_size": 100,
    "stride": 50,

    # groups to use (센서-모달리티 단위). 데이터셋에 없으면 자동 skip
    "groups": ["chest_acc", "ankle_acc", "arm_acc", "ankle_gyro", "arm_gyro"],

    # training
    "batch_size": 64,
    "epochs": 20,
    "lr": 1e-3,
    "seed": 42,

    # model
    "latent_dim": 64,
    "hidden_dim": 128,

    # losses
    "lambda_recon": 1.0,
    "lambda_soft": 0.5,        # soft supervision: p_hat(t) ~= 1 - s_det(t)
    "lambda_pair": 0.3,        # consistent/inconsistent pair contrast
    "lambda_inertial": 0.2,    # inertial invariance

    # detector smoothing
    "det_savgol": True,
    "det_savgol_win": 11,
    "det_savgol_poly": 2,

    # detector robust normalization
    "mad_eps": 1e-6,
    "sigmoid_tau": 1.5,

    # fusion quality (self-consistency)
    "quality_quantile": 0.2,   # top/bottom 20%
    "quality_pairs": 128,      # random pairs to estimate quality
    "quality_tau": 1.0,        # softmax temperature for weights

    # pair loss sampling
    "pair_delta": 5,           # pair offset in time steps inside window
    "pair_margin": 0.2,        # negative margin for similarity
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])


# =========================================================
# 1) MHEALTH LOADING (multi-group, missing-friendly)
# =========================================================
def load_mhealth_df(data_dir: str, target_activities):
    log_files = glob.glob(os.path.join(data_dir, "*.log"))
    if not log_files:
        raise FileNotFoundError(f"No .log files found in {data_dir}")

    all_rows = []
    for fpath in log_files:
        filename = os.path.basename(fpath)
        try:
            sub_id = int("".join(filter(str.isdigit, filename)))
        except:
            sub_id = 0

        try:
            df = pd.read_csv(fpath, sep=r"\s+", header=None, engine="python")
        except:
            df = pd.read_csv(fpath, sep="\t", header=None)

        # Standard MHEALTH indexing (0-based)
        # 0-2 chest acc
        # 5-7 ankle acc
        # 8-10 ankle gyro
        # 14-16 arm acc
        # 17-19 arm gyro
        # 23 label
        if df.shape[1] <= 23:
            raise ValueError("Unexpected MHEALTH column count.")

        df = df.copy()
        df["label"] = df.iloc[:, 23].astype(int)
        df["subject_id"] = sub_id

        df = df[df["label"].isin(target_activities)].copy()
        if len(df) > 0:
            all_rows.append(df)

    if not all_rows:
        raise ValueError("No data found for specified activities.")

    df_all = pd.concat(all_rows, ignore_index=True)
    print(f"[Data] Total samples: {len(df_all)} | Subjects: {sorted(df_all['subject_id'].unique())} | Acts: {sorted(df_all['label'].unique())}")
    return df_all


def get_group_array_from_block(block_np: np.ndarray, group: str):
    """
    block_np: (T, 24) raw columns from MHEALTH log row (0..23)
    returns (T,3) or None if group not available
    """
    if group == "chest_acc":
        return block_np[:, 0:3]
    if group == "ankle_acc":
        return block_np[:, 5:8]
    if group == "arm_acc":
        return block_np[:, 14:17]
    if group == "ankle_gyro":
        return block_np[:, 8:11]
    if group == "arm_gyro":
        return block_np[:, 17:20]
    # chest_gyro does not exist in MHEALTH => None
    return None


def create_blocks_by_subject_activity(df_all: pd.DataFrame):
    """
    Simple block extraction: subject x activity => one long sequence.
    (원하면 여기서 contiguous segment split로 더 정교하게 바꿀 수 있음)
    Returns list of dict:
      { "subject": int, "act": int, "raw": np.ndarray (T,24) }
    """
    blocks = []
    # df_all currently has extra cols label/subject_id at end; keep original 0..23 in numpy
    # We'll rebuild raw 24-col matrix from df row values [0..23]
    base_cols = list(range(24))
    # df_all columns are 0..23 plus label/subject_id; ensure we can index numeric columns
    for sub in sorted(df_all["subject_id"].unique()):
        sub_df = df_all[df_all["subject_id"] == sub]
        for act in sorted(sub_df["label"].unique()):
            act_df = sub_df[sub_df["label"] == act]
            raw = act_df[base_cols].to_numpy(dtype=np.float32)
            if len(raw) >= CONFIG["window_size"]:
                blocks.append({"subject": sub, "act": act, "raw": raw})
    return blocks


def sliding_windows(arr: np.ndarray, win: int, stride: int):
    """arr: (T, C) -> list of (win, C)"""
    out = []
    for i in range(0, len(arr) - win + 1, stride):
        out.append(arr[i : i + win])
    return out


# =========================================================
# 2) Detector: group-wise change -> robust normalize -> steady score
# =========================================================
def robust_zscore(x: np.ndarray, eps=1e-6):
    med = np.median(x)
    mad = np.median(np.abs(x - med)) + eps
    return (x - med) / mad


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def acc_change_score(acc_3: np.ndarray, var_win: int = 10):
    """
    acc_3: (T,3)
    change components:
      (1) direction change
      (2) local direction variance
      (3) jerk (optional)
    returns c: (T,)
    """
    a = acc_3
    T = len(a)
    # direction unit vectors
    u = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)

    # (1) direction change (1 - cos) with prev
    cos_prev = np.sum(u[1:] * u[:-1], axis=1)
    cos_prev = np.clip(cos_prev, -1.0, 1.0)
    dir_change = np.zeros(T, dtype=np.float32)
    dir_change[1:] = 1.0 - cos_prev

    # (2) local direction variance (your 기존 코드와 같은 취지)
    half = var_win // 2
    dir_var = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        local = u[s:e]
        dir_var[i] = float(np.mean(np.var(local, axis=0)))

    # (3) jerk magnitude
    jerk = np.zeros(T, dtype=np.float32)
    jerk[1:] = np.linalg.norm(a[1:] - a[:-1], axis=1)

    # combine (weights can be tuned; start simple)
    c = dir_change + dir_var + 0.3 * jerk
    return c


def gyro_change_score(gyro_3: np.ndarray, var_win: int = 10):
    """
    gyro_3: (T,3)
    change components:
      (1) delta omega magnitude OR vector delta
      (2) local variance of magnitude
      (3) magnitude (weak)
    returns c: (T,)
    """
    w = gyro_3
    T = len(w)
    mag = np.linalg.norm(w, axis=1)

    # (1) change
    dmag = np.zeros(T, dtype=np.float32)
    dmag[1:] = np.abs(mag[1:] - mag[:-1])

    # (2) local variance
    half = var_win // 2
    lvar = np.zeros(T, dtype=np.float32)
    for i in range(T):
        s = max(0, i - half)
        e = min(T, i + half + 1)
        lvar[i] = float(np.var(mag[s:e]))

    # (3) magnitude (weak)
    c = dmag + lvar + 0.2 * mag
    return c


def steady_score_from_change(c: np.ndarray, tau: float, eps: float, do_savgol: bool, win: int, poly: int):
    """
    threshold-free:
      c -> robust z -> sigmoid -> steady score in [0,1]
    """
    cz = robust_zscore(c, eps=eps)
    # larger change => smaller steady
    s = 1.0 - sigmoid(cz / tau)

    if do_savgol and len(s) >= win and (win % 2 == 1):
        s = savgol_filter(s, window_length=win, polyorder=poly).astype(np.float32)
        s = np.clip(s, 0.0, 1.0)
    return s.astype(np.float32)


def compute_group_steady_score(group_key: str, Xg: np.ndarray, cfg):
    """
    group_key: e.g. chest_acc / ankle_gyro
    Xg: (T,3)
    returns steady score s_g: (T,)
    """
    if Xg is None:
        return None

    if "acc" in group_key:
        c = acc_change_score(Xg, var_win=10)
    elif "gyro" in group_key:
        c = gyro_change_score(Xg, var_win=10)
    else:
        # unknown group type -> fallback
        c = np.linalg.norm(np.diff(Xg, axis=0, prepend=Xg[:1]), axis=1)

    s = steady_score_from_change(
        c,
        tau=cfg["sigmoid_tau"],
        eps=cfg["mad_eps"],
        do_savgol=cfg["det_savgol"],
        win=cfg["det_savgol_win"],
        poly=cfg["det_savgol_poly"],
    )
    return s


# =========================================================
# 3) Self-consistency quality for sensor fusion
# =========================================================
def cosine_sim(a: np.ndarray, b: np.ndarray, eps=1e-8):
    na = np.linalg.norm(a) + eps
    nb = np.linalg.norm(b) + eps
    return float(np.dot(a, b) / (na * nb))


def estimate_self_consistency_quality(group_key: str, Xg: np.ndarray, sg: np.ndarray, cfg):
    """
    Quality idea:
      - top steady (high s) frames should be mutually similar
      - bottom steady (low s) frames should be less similar / less consistent
    Use raw group vectors for similarity:
      acc: direction vectors u_t
      gyro: omega vectors (or normalized omega)
    """
    if Xg is None or sg is None:
        return None

    T = len(sg)
    q = cfg["quality_quantile"]
    k_pairs = cfg["quality_pairs"]

    # indices for top/bottom quantile
    order = np.argsort(sg)
    n = max(5, int(q * T))
    low_idx = order[:n]
    high_idx = order[-n:]

    if len(low_idx) < 5 or len(high_idx) < 5:
        return 0.0

    # feature vectors used for similarity
    if "acc" in group_key:
        V = Xg / (np.linalg.norm(Xg, axis=1, keepdims=True) + 1e-8)  # direction
    else:
        V = Xg  # gyro vector itself is fine to start

    def mean_pair_sim(idxs):
        sims = []
        for _ in range(k_pairs):
            i, j = np.random.choice(idxs, size=2, replace=True)
            sims.append(cosine_sim(V[i], V[j]))
        return float(np.mean(sims))

    S_high = mean_pair_sim(high_idx)
    S_low = mean_pair_sim(low_idx)
    quality = S_high - S_low
    return quality


def fuse_group_scores(groups_dict, cfg):
    """
    groups_dict: { group_key: {"X": (T,3), "s": (T,), "q": float} }
    Returns:
      s_fused: (T,)
      weights: {group_key: w}
      qualities: {group_key: q}
    """
    avail = [(g, d) for g, d in groups_dict.items() if d["s"] is not None]
    if not avail:
        raise ValueError("No available groups for fusion.")

    qs = np.array([d["q"] for _, d in avail], dtype=np.float32)
    # softmax weights over quality
    tau = cfg["quality_tau"]
    ws = np.exp(qs / max(tau, 1e-6))
    ws = ws / (ws.sum() + 1e-8)

    T = len(avail[0][1]["s"])
    s_fused = np.zeros(T, dtype=np.float32)
    weights = {}
    qualities = {}
    for (g, d), w in zip(avail, ws):
        s_fused += float(w) * d["s"]
        weights[g] = float(w)
        qualities[g] = float(d["q"])

    s_fused = np.clip(s_fused, 0.0, 1.0)
    return s_fused, weights, qualities


# =========================================================
# 4) Dataset: returns x (C,T), steady score s(t), transition pseudo y(t)=1-s(t)
# =========================================================
class MainEngineDataset(Dataset):
    def __init__(self, blocks, cfg):
        self.cfg = cfg
        self.groups = cfg["groups"]
        self.win = cfg["window_size"]
        self.stride = cfg["stride"]

        self.samples = []  # list of dict: {"x": (C,T), "s": (T,), "y": (T,), "meta": {...}}

        for b in blocks:
            raw = b["raw"]  # (T,24)
            subject = b["subject"]
            act = b["act"]

            # For each group, get full-seq array
            full_groups = {}
            for g in self.groups:
                Xg = get_group_array_from_block(raw, g)
                if Xg is None:
                    full_groups[g] = None
                else:
                    full_groups[g] = Xg.astype(np.float32)

            # Create windows over time using a reference length (raw length)
            T = len(raw)
            for st in range(0, T - self.win + 1, self.stride):
                ed = st + self.win
                groups_dict = {}
                x_parts = []

                # compute group steady scores inside this window
                for g in self.groups:
                    Xg_full = full_groups.get(g, None)
                    if Xg_full is None:
                        Xw = None
                        sw = None
                        qw = None
                    else:
                        Xw = Xg_full[st:ed]  # (win,3)
                        sw = compute_group_steady_score(g, Xw, cfg)  # (win,)
                        qw = estimate_self_consistency_quality(g, Xw, sw, cfg)
                    groups_dict[g] = {"X": Xw, "s": sw, "q": (0.0 if qw is None else qw)}
                    if Xw is not None:
                        x_parts.append(Xw)

                if len(x_parts) == 0:
                    continue

                # fuse steady scores with self-consistency quality weights
                s_fused, w_dict, q_dict = fuse_group_scores(groups_dict, cfg)
                y_pseudo = (1.0 - s_fused).astype(np.float32)  # transition likelihood

                # input tensor: concat available group signals along channel
                Xcat = np.concatenate(x_parts, axis=1)  # (win, C)
                # standardize per-window (simple, robust enough for start)
                mu = Xcat.mean(axis=0, keepdims=True)
                sd = Xcat.std(axis=0, keepdims=True) + 1e-6
                Xcat = (Xcat - mu) / sd

                self.samples.append({
                    "x": Xcat.astype(np.float32),     # (win,C)
                    "s": s_fused.astype(np.float32),  # (win,)
                    "y": y_pseudo.astype(np.float32), # (win,)
                    "meta": {"subject": subject, "act": act, "weights": w_dict, "qualities": q_dict}
                })

        print(f"[Dataset] samples/windows = {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        d = self.samples[idx]
        x = torch.tensor(d["x"], dtype=torch.float32).transpose(0, 1)  # (C,T)
        s = torch.tensor(d["s"], dtype=torch.float32)                  # (T,)
        y = torch.tensor(d["y"], dtype=torch.float32)                  # (T,)
        return x, s, y


# =========================================================
# 5) Model: latent sequence + transition prob + recon
# =========================================================
class MainEngineNet(nn.Module):
    def __init__(self, input_ch: int, hidden_dim: int, latent_dim: int, win: int):
        super().__init__()
        self.win = win
        self.input_ch = input_ch

        self.backbone = nn.Sequential(
            nn.Conv1d(input_ch, hidden_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
        )

        # latent sequence (B, D, T)
        self.to_latent = nn.Sequential(
            nn.Conv1d(hidden_dim, latent_dim, kernel_size=1),
            nn.ReLU(),
        )

        # transition probability head (B,1,T)
        self.trans_head = nn.Sequential(
            nn.Conv1d(latent_dim, 32, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(32, 1, kernel_size=1),
        )

        # inertial projection (for correlation with gravity direction)
        self.grav_proj = nn.Linear(latent_dim, 3)

        # reconstruction decoder (autoencoder)
        self.decoder = nn.Sequential(
            nn.Conv1d(latent_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, input_ch, kernel_size=3, padding=1),
        )

    def forward(self, x):
        # x: (B,C,T)
        h = self.backbone(x)          # (B,H,T)
        z = self.to_latent(h)         # (B,D,T)
        logits = self.trans_head(z)   # (B,1,T)
        p = torch.sigmoid(logits)     # (B,1,T)
        x_recon = self.decoder(z)     # (B,C,T)
        return z, p, x_recon


# =========================================================
# 6) Losses: recon + soft transition supervision + pair contrast + inertial invariance
# =========================================================
def weighted_soft_bce(p_hat, y_soft, eps=1e-6):
    """
    p_hat: (B,1,T) in [0,1]
    y_soft: (B,T) in [0,1]
    """
    p = p_hat.squeeze(1)
    y = y_soft
    # standard BCE with soft targets
    return F.binary_cross_entropy(p.clamp(eps, 1-eps), y.clamp(eps, 1-eps))


def pair_consistency_loss(z, s, cfg):
    """
    z: (B,D,T)
    s: (B,T) steady score from detector
    Define positive weight = s_t * s_{t+Δ}
    Define negative weight = (1-s_t) * (1-s_{t+Δ})
    Similarity = cosine(z_t, z_{t+Δ})
    Loss:
      pos: w_pos * (1 - sim)
      neg: w_neg * relu(sim - margin)
    """
    B, D, T = z.shape
    delta = cfg["pair_delta"]
    margin = cfg["pair_margin"]

    if T <= delta:
        return torch.tensor(0.0, device=z.device)

    z1 = z[:, :, :-delta]               # (B,D,T-d)
    z2 = z[:, :, delta:]                # (B,D,T-d)
    s1 = s[:, :-delta]                  # (B,T-d)
    s2 = s[:, delta:]                   # (B,T-d)

    # cosine similarity along D
    z1n = F.normalize(z1, dim=1)
    z2n = F.normalize(z2, dim=1)
    sim = (z1n * z2n).sum(dim=1)        # (B,T-d) in [-1,1]

    w_pos = (s1 * s2).detach()
    w_neg = ((1 - s1) * (1 - s2)).detach()

    pos_loss = (w_pos * (1.0 - sim)).mean()
    neg_loss = (w_neg * F.relu(sim - margin)).mean()
    return pos_loss + neg_loss


def inertial_invariance_loss(z, x, s, cfg):
    """
    Make latent for steady parts less aligned with gravity direction.
    - get gravity direction from acc channels that exist in x.
    Here we approximate gravity direction using mean of the first 3 channels
    IF those correspond to some acc. Since x is concatenated multi-groups, we
    instead compute gravity direction from the average of all acc-like channels:
      Take every 3 channels block as a vector, average them.
    (This is a simple baseline; you can refine by using known group indices.)
    """
    B, D, T = z.shape
    C = x.shape[1]

    # estimate "gravity direction" from x by averaging 3-axis chunks
    # (assumes concatenation keeps (win,3) blocks; true in our dataset)
    nvec = C // 3
    if nvec == 0:
        return torch.tensor(0.0, device=z.device)

    x_reshaped = x[:, :nvec*3, :].reshape(B, nvec, 3, T)  # (B,nvec,3,T)
    g = x_reshaped.mean(dim=3).mean(dim=1)                # (B,3)
    g = F.normalize(g, dim=1)

    # steady-weighted mean latent
    w = s / (s.sum(dim=1, keepdim=True) + 1e-6)          # (B,T)
    z_bar = (z * w.unsqueeze(1)).sum(dim=2)              # (B,D)

    # project latent to 3D and reduce correlation with g
    z3 = cfg["_grav_proj"](z_bar)                         # (B,3)
    z3 = F.normalize(z3, dim=1)
    corr = torch.abs(F.cosine_similarity(z3, g, dim=1)).mean()
    return corr


# =========================================================
# 7) Train / Eval
# =========================================================
def train_one_fold(model, loader, cfg):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    model.train()
    for ep in range(cfg["epochs"]):
        L = []
        for x, s, y in loader:
            x = x.to(DEVICE)
            s = s.to(DEVICE)
            y = y.to(DEVICE)

            z, p, x_recon = model(x)

            loss_recon = F.mse_loss(x_recon, x)
            loss_soft  = weighted_soft_bce(p, y)

            loss_pair  = pair_consistency_loss(z, s, cfg)

            # inertial loss uses model.grav_proj; pass it through cfg hook for simplicity
            cfg["_grav_proj"] = model.grav_proj
            loss_inert = inertial_invariance_loss(z, x, s, cfg)

            loss = (cfg["lambda_recon"] * loss_recon +
                    cfg["lambda_soft"]  * loss_soft +
                    cfg["lambda_pair"]  * loss_pair +
                    cfg["lambda_inertial"] * loss_inert)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            L.append([loss.item(), loss_recon.item(), loss_soft.item(), loss_pair.item(), loss_inert.item()])

        sched.step()
        if (ep + 1) == cfg["epochs"]:
            arr = np.array(L)
            print(f"  > Epoch {ep+1:02d} | Loss {arr[:,0].mean():.4f} | Recon {arr[:,1].mean():.4f} | Soft {arr[:,2].mean():.4f} | Pair {arr[:,3].mean():.4f} | Inert {arr[:,4].mean():.4f}")

    return model


@torch.no_grad()
def eval_one_fold(model, loader):
    model.eval()
    # We evaluate: how well p_hat matches pseudo y (sanity) + distributions.
    all_p = []
    all_y = []
    for x, s, y in loader:
        x = x.to(DEVICE)
        z, p, x_recon = model(x)
        all_p.append(p.squeeze(1).cpu().numpy())
        all_y.append(y.cpu().numpy())

    P = np.concatenate(all_p, axis=0)  # (N,T)
    Y = np.concatenate(all_y, axis=0)
    mae = float(np.mean(np.abs(P - Y)))
    mse = float(np.mean((P - Y) ** 2))
    return {"mae_soft": mae, "mse_soft": mse}


# =========================================================
# 8) Main (LOSO)
# =========================================================
def main():
    print("=" * 70)
    print("Main Engine (Threshold-free) - MHEALTH LOSO (per-activity)")
    print("=" * 70)

    df_all = load_mhealth_df(CONFIG["data_dir"], CONFIG["target_activities"])
    blocks_all = create_blocks_by_subject_activity(df_all)

    acts = sorted({b["act"] for b in blocks_all})  # [6,7,11]

    overall_results = []

    for act in acts:
        print("\n" + "#" * 70)
        print(f"[Activity {act}] Single-activity LOSO")
        print("#" * 70)

        blocks = [b for b in blocks_all if b["act"] == act]
        subjects = sorted({b["subject"] for b in blocks})
        print(f"[Blocks] act={act} total={len(blocks)} | folds={len(subjects)}")

        act_results = []

        for test_sub in subjects:
            print(f"\n--- Fold: act {act} | test subject {test_sub} ---")

            train_blocks = [b for b in blocks if b["subject"] != test_sub]
            test_blocks  = [b for b in blocks if b["subject"] == test_sub]

            train_ds = MainEngineDataset(train_blocks, CONFIG)
            test_ds  = MainEngineDataset(test_blocks, CONFIG)

            if len(train_ds) == 0 or len(test_ds) == 0:
                print("  [Skip] empty dataset in this fold.")
                continue

            train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True)
            test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False)

            x0, s0, y0 = train_ds[0]
            input_ch = x0.shape[0]

            model = MainEngineNet(
                input_ch=input_ch,
                hidden_dim=CONFIG["hidden_dim"],
                latent_dim=CONFIG["latent_dim"],
                win=CONFIG["window_size"]
            ).to(DEVICE)

            model = train_one_fold(model, train_loader, CONFIG)
            met = eval_one_fold(model, test_loader)
            print(f"  > Eval (sanity): MAE={met['mae_soft']:.4f} | MSE={met['mse_soft']:.4f}")

            act_results.append(met)         # fold 단위 저장
            overall_results.append(met)     # 전체도 같이 저장

        # act별 평균 출력
        if act_results:
            mae_a = np.mean([r["mae_soft"] for r in act_results])
            mse_a = np.mean([r["mse_soft"] for r in act_results])
            print(f"\n[Activity {act}] Avg MAE={mae_a:.4f} | Avg MSE={mse_a:.4f}")

    # 전체 평균
    if overall_results:
        mae = np.mean([r["mae_soft"] for r in overall_results])
        mse = np.mean([r["mse_soft"] for r in overall_results])
        print("\n" + "=" * 70)
        print("FINAL (sanity) LOSO RESULTS (overall)")
        print("=" * 70)
        print(f"Avg MAE(p_hat vs pseudo y): {mae:.4f}")
        print(f"Avg MSE(p_hat vs pseudo y): {mse:.4f}")

if __name__ == "__main__":
    main()


Main Engine (Threshold-free) - MHEALTH LOSO (per-activity)
[Data] Total samples: 88476 | Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)] | Acts: [np.int64(6), np.int64(7), np.int64(11)]

######################################################################
[Activity 6] Single-activity LOSO
######################################################################
[Blocks] act=6 total=10 | folds=10

--- Fold: act 6 | test subject 1 ---
[Dataset] samples/windows = 493
[Dataset] samples/windows = 60
  > Epoch 20 | Loss 0.5250 | Recon 0.0700 | Soft 0.6871 | Pair 0.2202 | Inert 0.2267
  > Eval (sanity): MAE=0.0602 | MSE=0.0056

--- Fold: act 6 | test subject 2 ---
[Dataset] samples/windows = 491
[Dataset] samples/windows = 62
  > Epoch 20 | Loss 0.5266 | Recon 0.0700 | Soft 0.6866 | Pair 0.2192 | Inert 0.2380
  > Eval (sanity): MAE=0.0522 | MSE=0.0042

--- Fold: act 6 | test subject 3 ---
[Dataset] sa